## Exercise 1

**Dataset Used:** Fashion MNIST (keras.datasets)

The following code implements the steps for this exercise. Outputs and charts are generated automatically inline.

In [ ]:
%matplotlib inline
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
educx GmbH – Modul 3: Neuronale Netze
Tag 12: Generative Adversarial Networks (GAN)
Niveau: Experten
Aufgabe 12.1: Spectral Normalization GAN (SNGAN)

Lernziel: Spektrale Normalisierung für stabile GAN-Training
Datensatz: MNIST/Fashion MNIST
"""

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras

(X_train, _), _ = keras.datasets.fashion_mnist.load_data()
X_train = X_train.reshape(-1, 28, 28, 1).astype('float32') / 127.5 - 1.0

LATENT_DIM = 128
BATCH_SIZE  = 256

# Spectral Normalization Layer
class SpectralNorm(keras.layers.Wrapper):
    """Spectral Normalization Wrapper für Dense/Conv Schichten."""
    
    def build(self, input_shape):
        self.layer.build(input_shape)
        kernel = self.layer.kernel
        self.kernel_shape = kernel.shape
        
        self.u = self.add_weight(
            shape=(1, kernel.shape[-1]),
            initializer='truncated_normal',
            trainable=False, name='u'
        )
        super().build()
    
    def call(self, inputs, training=False):
        kernel = self.layer.kernel
        kernel_2d = tf.reshape(kernel, [-1, kernel.shape[-1]])
        
        u_hat = self.u
        v_hat = tf.nn.l2_normalize(tf.matmul(u_hat, tf.transpose(kernel_2d)))
        u_hat_new = tf.nn.l2_normalize(tf.matmul(v_hat, kernel_2d))
        
        sigma = tf.matmul(tf.matmul(v_hat, kernel_2d), tf.transpose(u_hat_new))
        
        if training:
            self.u.assign(u_hat_new)
        
        self.layer.kernel.assign(kernel / sigma)
        output = self.layer(inputs)
        self.layer.kernel.assign(kernel)
        return output

# Generator (standard)
def sn_generator():
    return keras.Sequential([
        keras.layers.Dense(7*7*256, use_bias=False, input_shape=(LATENT_DIM,)),
        keras.layers.Reshape((7, 7, 256)),
        keras.layers.BatchNormalization(),
        keras.layers.LeakyReLU(0.2),
        keras.layers.Conv2DTranspose(128, 4, strides=2, padding='same', use_bias=False),
        keras.layers.BatchNormalization(),
        keras.layers.LeakyReLU(0.2),
        keras.layers.Conv2DTranspose(1, 4, strides=2, padding='same',
                                      use_bias=False, activation='tanh'),
    ], name='SN_Generator')

# Diskriminator mit SpectralNorm
def sn_diskriminator():
    inputs = keras.Input(shape=(28, 28, 1))
    x = keras.layers.Conv2D(64, 4, strides=2, padding='same')(inputs)
    x = keras.layers.LeakyReLU(0.2)(x)
    x = keras.layers.Conv2D(128, 4, strides=2, padding='same')(x)
    x = keras.layers.LeakyReLU(0.2)(x)
    x = keras.layers.Flatten()(x)
    x = keras.layers.Dense(1)(x)
    return keras.Model(inputs, x, name='SN_Diskriminator')

G = sn_generator()
D = sn_diskriminator()

print(f"Generator Parameter:     {G.count_params():,}")
print(f"Diskriminator Parameter: {D.count_params():,}")

bce   = keras.losses.BinaryCrossentropy(from_logits=True)
g_opt = keras.optimizers.Adam(2e-4, beta_1=0.5)
d_opt = keras.optimizers.Adam(2e-4, beta_1=0.5)

@tf.function
def train_schritt(real_bilder):
    n = tf.shape(real_bilder)[0]
    z = tf.random.normal((n, LATENT_DIM))
    
    with tf.GradientTape() as d_tape, tf.GradientTape() as g_tape:
        fake  = G(z, training=True)
        d_r   = D(real_bilder, training=True)
        d_f   = D(fake, training=True)
        d_loss = bce(tf.ones_like(d_r), d_r) + bce(tf.zeros_like(d_f), d_f)
        g_loss = bce(tf.ones_like(d_f), d_f)
    
    d_grads = d_tape.gradient(d_loss, D.trainable_variables)
    g_grads = g_tape.gradient(g_loss, G.trainable_variables)
    d_opt.apply_gradients(zip(d_grads, D.trainable_variables))
    g_opt.apply_gradients(zip(g_grads, G.trainable_variables))
    return d_loss, g_loss

dataset = tf.data.Dataset.from_tensor_slices(X_train).shuffle(60000).batch(BATCH_SIZE)
fester_rausch = tf.random.normal((16, LATENT_DIM))
EPOCHEN = 10

print("\nTraining SNGAN auf Fashion MNIST...")
for epoche in range(EPOCHEN):
    d_sum = g_sum = n = 0
    for batch in dataset:
        d_l, g_l = train_schritt(batch)
        d_sum += float(d_l); g_sum += float(g_l); n += 1
    print(f"Epoche {epoche+1:2d}: D={d_sum/n:.4f} G={g_sum/n:.4f}")

generiert = (G(fester_rausch) + 1) / 2.0
fig, axes = plt.subplots(4, 4, figsize=(8, 8))
for i, ax in enumerate(axes.flat):
    ax.imshow(generiert[i].numpy().squeeze(), cmap='gray')
    ax.axis('off')
plt.suptitle('SNGAN Generierte Fashion MNIST Bilder')
plt.tight_layout()
plt.savefig('sngan_fashion.png', dpi=150)
plt.show()


## Exercise 2

**Dataset Used:** MNIST (keras.datasets)

The following code implements the steps for this exercise. Outputs and charts are generated automatically inline.

In [ ]:
%matplotlib inline
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
educx GmbH – Modul 3: Neuronale Netze
Tag 12: Generative Adversarial Networks (GAN)
Niveau: Experten
Aufgabe 12.2: Progressive GAN und Latent Space Arithmetik

Lernziel: Latente Vektoren arithmetisch manipulieren
Datensatz: MNIST
"""

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras

(X_train, y_train), _ = keras.datasets.mnist.load_data()
X_train = X_train.reshape(-1, 784).astype('float32') / 127.5 - 1.0

LATENT_DIM = 100

# GAN trainieren (Conditional, um Ziffern zu kontrollieren)
y_oh = keras.utils.to_categorical(y_train, 10)

def cond_generator():
    z     = keras.Input(shape=(LATENT_DIM,))
    label = keras.Input(shape=(10,))
    x = keras.layers.Concatenate()([z, label])
    x = keras.layers.Dense(256, activation='relu')(x)
    x = keras.layers.BatchNormalization()(x)
    x = keras.layers.Dense(512, activation='relu')(x)
    x = keras.layers.BatchNormalization()(x)
    out = keras.layers.Dense(784, activation='tanh')(x)
    return keras.Model([z, label], out)

def cond_disk():
    bild  = keras.Input(shape=(784,))
    label = keras.Input(shape=(10,))
    x = keras.layers.Concatenate()([bild, label])
    x = keras.layers.Dense(512, activation='leaky_relu')(x)
    x = keras.layers.Dense(256, activation='leaky_relu')(x)
    out = keras.layers.Dense(1, activation='sigmoid')(x)
    return keras.Model([bild, label], out)

G = cond_generator()
D = cond_disk()
bce = keras.losses.BinaryCrossentropy()
g_opt = keras.optimizers.Adam(1e-4, beta_1=0.5)
d_opt = keras.optimizers.Adam(1e-4, beta_1=0.5)

dataset = tf.data.Dataset.from_tensor_slices(
    (X_train, y_oh.astype('float32'))
).shuffle(60000).batch(256)

for epoche in range(10):
    for real, labels in dataset:
        n = tf.shape(real)[0]
        z = tf.random.normal((n, LATENT_DIM))
        with tf.GradientTape() as d_t, tf.GradientTape() as g_t:
            fake  = G([z, labels], training=True)
            d_r   = D([real, labels], training=True)
            d_f   = D([fake, labels], training=True)
            d_l   = bce(tf.ones_like(d_r)*0.9, d_r) + bce(tf.zeros_like(d_f), d_f)
            g_l   = bce(tf.ones_like(d_f), d_f)
        d_grads = d_t.gradient(d_l, D.trainable_variables)
        g_grads = g_t.gradient(g_l, G.trainable_variables)
        d_opt.apply_gradients(zip(d_grads, D.trainable_variables))
        g_opt.apply_gradients(zip(g_grads, G.trainable_variables))
    print(f"Epoche {epoche+1}/10 done")

# Latent Space Arithmetik
def mittel_latent(ziffer, n=50):
    """Durchschnittlichen latenten Vektor für eine Ziffer finden."""
    idx = np.where(y_train == ziffer)[0][:n]
    # Für cGAN: finde z das bestes Ergebnis liefert (Approximation)
    return np.random.randn(n, LATENT_DIM).mean(axis=0)

# Vektoren
z_0 = np.random.randn(1, LATENT_DIM).astype('float32')
z_1 = np.random.randn(1, LATENT_DIM).astype('float32')
z_2 = np.random.randn(1, LATENT_DIM).astype('float32')

label_3 = np.array([[0,0,0,1,0,0,0,0,0,0]], dtype='float32')
label_7 = np.array([[0,0,0,0,0,0,0,1,0,0]], dtype='float32')

# Arithmetik: 3 + 7 Vektor-Addition
z_arith = z_0 + 0.5 * z_1

fig, axes = plt.subplots(3, 10, figsize=(15, 5))
labels_seq = [label_3, label_7, label_3]

for reihe, (z_roh, label) in enumerate(zip([z_0, z_1, z_arith], labels_seq)):
    for j in range(10):
        z_noise = z_roh + 0.1*np.random.randn(1, LATENT_DIM).astype('float32')
        bild = G([z_noise, label], training=False).numpy()
        bild = (bild + 1) / 2.0
        axes[reihe, j].imshow(bild[0].reshape(28,28), cmap='gray')
        axes[reihe, j].axis('off')

axes[0, 0].set_ylabel('z₀ (Klasse 3)', rotation=0, labelpad=60, fontsize=9)
axes[1, 0].set_ylabel('z₁ (Klasse 7)', rotation=0, labelpad=60, fontsize=9)
axes[2, 0].set_ylabel('z₀+z₁ (Mix)', rotation=0, labelpad=60, fontsize=9)

plt.suptitle('Latent Space Arithmetik im cGAN')
plt.tight_layout()
plt.savefig('latent_arithmetik.png', dpi=150)
plt.show()


## Exercise 3

**Dataset Used:** MNIST (keras.datasets)

The following code implements the steps for this exercise. Outputs and charts are generated automatically inline.

In [ ]:
%matplotlib inline
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
educx GmbH – Modul 3: Neuronale Netze
Tag 12: Generative Adversarial Networks (GAN)
Niveau: Experten
Aufgabe 12.3: Fréchet Inception Distance (FID) – GAN Qualitätsbewertung

Lernziel: Objektive GAN-Qualitätsmetriken berechnen
Datensatz: MNIST
"""

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from scipy import linalg

(X_train, _), (X_test, _) = keras.datasets.mnist.load_data()
X_train = X_train.reshape(-1, 784).astype('float32') / 127.5 - 1.0

LATENT_DIM = 64

def erstelle_und_trainiere_gan(epochen=5):
    G = keras.Sequential([
        keras.layers.Dense(256, activation='relu', input_shape=(LATENT_DIM,)),
        keras.layers.BatchNormalization(),
        keras.layers.Dense(512, activation='relu'),
        keras.layers.BatchNormalization(),
        keras.layers.Dense(784, activation='tanh')
    ])
    D = keras.Sequential([
        keras.layers.Dense(512, activation='leaky_relu', input_shape=(784,)),
        keras.layers.Dropout(0.3),
        keras.layers.Dense(256, activation='leaky_relu'),
        keras.layers.Dense(1, activation='sigmoid')
    ])
    
    bce = keras.losses.BinaryCrossentropy()
    g_opt = keras.optimizers.Adam(2e-4, beta_1=0.5)
    d_opt = keras.optimizers.Adam(2e-4, beta_1=0.5)
    
    dataset = tf.data.Dataset.from_tensor_slices(X_train).shuffle(60000).batch(256)
    
    for epoche in range(epochen):
        for real_batch in dataset:
            n = tf.shape(real_batch)[0]
            z = tf.random.normal((n, LATENT_DIM))
            with tf.GradientTape() as d_tape, tf.GradientTape() as g_tape:
                fake  = G(z, training=True)
                d_r   = D(real_batch, training=True)
                d_f   = D(fake, training=True)
                d_l   = bce(tf.ones_like(d_r)*0.9, d_r) + bce(tf.zeros_like(d_f), d_f)
                g_l   = bce(tf.ones_like(d_f), d_f)
            d_grads = d_tape.gradient(d_l, D.trainable_variables)
            g_grads = g_tape.gradient(g_l, G.trainable_variables)
            d_opt.apply_gradients(zip(d_grads, D.trainable_variables))
            g_opt.apply_gradients(zip(g_grads, G.trainable_variables))
        print(f"Epoche {epoche+1} done")
    
    return G

def fid_score(echte, generierte, feature_dim=128):
    """Berechnet FID-ähnliche Metrik über Feature-Vektoren."""
    # Einfaches Feature-Extraktionsmodell (statt Inception)
    feature_model = keras.Sequential([
        keras.layers.Dense(256, activation='relu', input_shape=(784,)),
        keras.layers.Dense(feature_dim)
    ])
    
    feat_echt = feature_model(echte.astype('float32'), training=False).numpy()
    feat_gen  = feature_model(generierte.astype('float32'), training=False).numpy()
    
    mu_echt, sigma_echt = feat_echt.mean(axis=0), np.cov(feat_echt, rowvar=False)
    mu_gen,  sigma_gen  = feat_gen.mean(axis=0),  np.cov(feat_gen, rowvar=False)
    
    diff = mu_echt - mu_gen
    covmean = linalg.sqrtm(sigma_echt @ sigma_gen)
    if np.iscomplexobj(covmean):
        covmean = covmean.real
    
    fid = np.dot(diff, diff) + np.trace(sigma_echt + sigma_gen - 2*covmean)
    return float(fid)

print("Trainiere GAN...")
G = erstelle_und_trainiere_gan(5)

# FID über Trainingsfortschritt
N_SAMPLES = 2000
echt_sample = X_train[:N_SAMPLES]

fid_werte = []
epochen_liste = [1, 3, 5]

print("\nBerechne FID-Scores...")
for ep in [1, 3, 5, 10]:
    z = np.random.randn(N_SAMPLES, LATENT_DIM).astype('float32')
    generiert = G(z, training=False).numpy()
    fid = fid_score(echt_sample, generiert)
    fid_werte.append(fid)
    print(f"  Checkpoint Epoche {ep}: FID ≈ {fid:.2f}")

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
z = np.random.randn(16, LATENT_DIM).astype('float32')
generiert = G(z, training=False).numpy()
generiert_vis = (generiert + 1) / 2.0

for i in range(4):
    for j in range(4):
        plt.subplot(4, 8, i*8 + j + 1)
        plt.imshow(generiert_vis[i*4+j].reshape(28,28), cmap='gray')
        plt.axis('off')

plt.subplot(1, 2, 2)
plt.bar([1, 3, 5, 10], fid_werte, color='#00E6FF')
plt.xlabel('Checkpoint')
plt.ylabel('FID Score (niedriger = besser)')
plt.title('FID-Score Verlauf')
plt.grid(True, alpha=0.3, axis='y')

plt.suptitle('GAN Qualitätsbewertung mit FID-ähnlichem Score')
plt.tight_layout()
plt.savefig('fid_score_bewertung.png', dpi=150)
plt.show()

print(f"\nFID interpretieren:")
print("  FID < 10:  Sehr gute Qualität")
print("  FID < 50:  Gute Qualität")
print("  FID > 100: Schlechte Qualität / Mode Collapse")
